In [3]:
import sqlite3
import pandas as pd
import numpy as np

# Connect to the same database used for the demand model
conn = sqlite3.connect('/Users/manushdesai/Desktop/multi-agent/data/ecommerce.db')

# We JOIN returns with orders (to get order value), products (to get category),
# and customers (to get the persisted is_repeat_offender flag) --
# these give us the real signals for judging return risk
returns_df = pd.read_sql("""
    SELECT r.*, o.total_amount, p.category, c.is_repeat_offender
    FROM returns r
    JOIN orders o ON r.order_id = o.order_id
    JOIN products p ON r.product_id = p.product_id
    JOIN customers c ON r.customer_id = c.customer_id
""", conn)

print("Total return records:", len(returns_df))
print("Flagged (suspicious):", returns_df['is_flagged'].sum())
print("Not flagged:", len(returns_df) - returns_df['is_flagged'].sum())

returns_df.head()

Total return records: 575
Flagged (suspicious): 206
Not flagged: 369


,return_id,order_id,customer_id,product_id,return_date,reason,days_since_order,customer_total_returns,is_flagged,agent_decision,total_amount,category,is_repeat_offender
0,1,1,125,32,2026-05-10,Wrong size / doesn't fit,1,1,0,None,5530.94,Apparel,0
1,2,15,96,64,2026-05-28,Wrong size / doesn't fit,8,1,0,None,15496.88,Sports,0
2,3,19,114,79,2026-03-22,Received wrong item,29,1,0,None,30791.37,Electronics,0
3,4,29,62,78,2026-04-28,Item not as described,37,1,0,None,5428.80,Home & Kitchen,0
4,5,43,94,68,2026-06-19,Item damaged on arrival,1,1,0,None,21608.96,Electronics,0


In [4]:
# Models only understand numbers, not text like "Electronics" or "Beauty" --
# convert category into 0/1 columns, same approach as the demand model
returns_df = pd.get_dummies(returns_df, columns=['category'], drop_first=True)

# These are the signals we believe are genuinely predictive of return risk:
# - days_since_order: very late returns are more suspicious
# - customer_total_returns: repeat returners are riskier
# - total_amount: high-value returns may warrant more scrutiny
# - is_repeat_offender: persisted flag from the customers table
# - category columns: some categories (Apparel/Beauty) naturally return more
feature_cols = (
    ['days_since_order', 'customer_total_returns', 'total_amount', 'is_repeat_offender']
    + [c for c in returns_df.columns if c.startswith('category_')]
)

X = returns_df[feature_cols]
y = returns_df['is_flagged']

print("Features used:", feature_cols)
print("Feature matrix shape:", X.shape)

Features used: ['days_since_order', 'customer_total_returns', 'total_amount', 'is_repeat_offender', 'category_Beauty', 'category_Electronics', 'category_Home & Kitchen', 'category_Sports']
Feature matrix shape: (575, 8)


In [5]:
from sklearn.model_selection import train_test_split

# stratify=y keeps the same flagged/not-flagged RATIO in both train and test sets.
# This matters more here than for the demand model, because with ~588 rows
# a random split could accidentally put too few (or too many) flagged cases
# in the test set, making evaluation unreliable.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train rows:", len(X_train), " | Flagged in train:", y_train.sum())
print("Test rows:", len(X_test), " | Flagged in test:", y_test.sum())

Train rows: 431  | Flagged in train: 154
Test rows: 144  | Flagged in test: 52


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Baseline: Logistic Regression. class_weight='balanced' tells it to pay
# more attention to the minority (flagged) class during training, since
# otherwise it could get lazy and just predict "not flagged" for everything.
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
log_reg.fit(X_train, y_train)
lr_pred = log_reg.predict(X_test)

# Our real model: Random Forest Classifier.
# max_depth kept shallow (5) -- with only ~440 training rows, a deep tree
# risks memorizing individual cases instead of learning general patterns.
rf_clf = RandomForestClassifier(n_estimators=150, max_depth=5, random_state=42, class_weight='balanced')
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)

print("Both models trained.")

Both models trained.


In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

def evaluate(name, y_true, y_pred):
    # We use precision/recall instead of plain accuracy because flagged
    # cases are the MINORITY class -- a model that just predicts "not
    # flagged" every time would still get high accuracy while being useless.
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print(f"{name:25s}  Precision: {precision:.2f}   Recall: {recall:.2f}   F1: {f1:.2f}")

print("--- Performance on TEST set ---")
evaluate("Logistic Regression", y_test, lr_pred)
evaluate("Random Forest", y_test, rf_pred)

print("\n--- Full classification report (Random Forest) ---")
print(classification_report(y_test, rf_pred, target_names=['Not flagged', 'Flagged']))

--- Performance on TEST set ---
Logistic Regression        Precision: 0.58   Recall: 0.60   F1: 0.59
Random Forest              Precision: 0.67   Recall: 0.54   F1: 0.60

--- Full classification report (Random Forest) ---
              precision    recall  f1-score   support

 Not flagged       0.76      0.85      0.80        92
     Flagged       0.67      0.54      0.60        52

    accuracy                           0.74       144
   macro avg       0.72      0.69      0.70       144
weighted avg       0.73      0.74      0.73       144



In [8]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# With ~588 rows, a single train/test split can be a bit unstable --
# a different random split could give noticeably different numbers.
# 5-fold cross-validation trains/tests 5 times on different slices and
# averages the result, giving a more reliable performance estimate.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_precision = cross_val_score(rf_clf, X, y, cv=cv, scoring='precision')
cv_recall = cross_val_score(rf_clf, X, y, cv=cv, scoring='recall')

print("Cross-validated Precision (5 folds):", np.round(cv_precision, 2))
print("Mean Precision:", round(cv_precision.mean(), 2))
print()
print("Cross-validated Recall (5 folds):", np.round(cv_recall, 2))
print("Mean Recall:", round(cv_recall.mean(), 2))

Cross-validated Precision (5 folds): [0.65 0.69 0.64 0.6  0.62]
Mean Precision: 0.64

Cross-validated Recall (5 folds): [0.49 0.61 0.66 0.59 0.5 ]
Mean Recall: 0.57


In [9]:
importances = pd.Series(rf_clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("--- Feature importance (Random Forest Classifier) ---")
print(importances)

--- Feature importance (Random Forest Classifier) ---
customer_total_returns     0.421429
days_since_order           0.244108
total_amount               0.174528
is_repeat_offender         0.087151
category_Electronics       0.023986
category_Home & Kitchen    0.020039
category_Sports            0.015511
category_Beauty            0.013249
dtype: float64


In [10]:
import joblib
import os

save_dir = '/Users/manushdesai/Desktop/multi-agent/models'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(rf_clf, f'{save_dir}/returns_model.pkl')

with open(f'{save_dir}/returns_model_features.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print("Model saved to:", save_dir)

Model saved to: /Users/manushdesai/Desktop/multi-agent/models
